# ClipForge · Motor editorial en Google Colab
Este cuaderno inicia un modelo Qwen local en la sesión gratuita de Colab y publica una URL temporal protegida por una clave. No requiere tarjeta ni clave de API de pago. La sesión se apaga al cerrarla o cuando Colab la desconecta; no es un servicio permanente.

**Recomendado:** Runtime → Change runtime type → T4 GPU (si está disponible). Ejecuta las celdas en orden. No compartas la URL ni el token que aparecen al final.

In [ ]:
# 1) Comprobar acelerador disponible
import subprocess, os, shutil, time, secrets, textwrap, pathlib, json
gpu = subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True,text=True)
print(gpu.stdout[:2500] if gpu.returncode == 0 else 'No hay GPU asignada. Se intentará CPU; será mucho más lento.')


In [ ]:
# 2) Instalar Ollama y Cloudflare Tunnel
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Instalación terminada')


In [ ]:
# 3) Arrancar Ollama, descargar el modelo y crear un proxy protegido
import subprocess, os, time, secrets, pathlib
os.environ['OLLAMA_HOST']='127.0.0.1:11434'
ollama = subprocess.Popen(['ollama','serve'],stdout=open('/tmp/ollama.log','w'),stderr=subprocess.STDOUT,env=os.environ.copy())
for _ in range(60):
    if subprocess.run(['bash','-lc','curl -fsS http://127.0.0.1:11434/api/tags >/dev/null'],capture_output=True).returncode == 0: break
    time.sleep(2)
subprocess.run(['ollama','pull','qwen2.5:3b-instruct-q4_K_M'],check=True)
TOKEN = secrets.token_urlsafe(32)
proxy = r'''from http.server import BaseHTTPRequestHandler, HTTPServer
import json, urllib.request, os
TOKEN = os.environ['CLIPFORGE_TOKEN']
class H(BaseHTTPRequestHandler):
 def do_OPTIONS(self):
  self.send_response(204); self.send_header('Access-Control-Allow-Origin','*'); self.send_header('Access-Control-Allow-Headers','Authorization, Content-Type'); self.send_header('Access-Control-Allow-Methods','POST, OPTIONS'); self.end_headers()
 def do_POST(self):
  if self.path != '/api/chat' or self.headers.get('Authorization') != 'Bearer '+TOKEN:
   self.send_response(401); self.end_headers(); return
  n=int(self.headers.get('Content-Length','0')); body=self.rfile.read(n)
  req=urllib.request.Request('http://127.0.0.1:11434/api/chat',data=body,headers={'Content-Type':'application/json'},method='POST')
  try:
   with urllib.request.urlopen(req,timeout=300) as r: out=r.read()
   self.send_response(200); self.send_header('Content-Type','application/json'); self.send_header('Access-Control-Allow-Origin','*'); self.end_headers(); self.wfile.write(out)
  except Exception as e:
   self.send_response(502); self.send_header('Access-Control-Allow-Origin','*'); self.end_headers(); self.wfile.write(str(e).encode())
HTTPServer(('127.0.0.1',8000),H).serve_forever()
'''
pathlib.Path('/tmp/clipforge_proxy.py').write_text(proxy)
env=os.environ.copy(); env['CLIPFORGE_TOKEN']=TOKEN
server=subprocess.Popen(['python','/tmp/clipforge_proxy.py'],env=env,stdout=open('/tmp/proxy.log','w'),stderr=subprocess.STDOUT)
print('Modelo cargado. Proxy protegido listo.')


In [ ]:
# 4) Abrir túnel temporal seguro
tunnel = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
import re, threading
lines=[]
def read_tunnel():
    for line in tunnel.stdout:
        lines.append(line)
threading.Thread(target=read_tunnel,daemon=True).start()
url=None
for _ in range(90):
    text=''.join(lines)
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if m: url=m.group(0); break
    time.sleep(1)
if not url: print('No se pudo crear el túnel. Revisa la salida de cloudflared:', ''.join(lines[-20:]))
else:
    print('\nURL DEL MOTOR:',url+'/api/chat')
    print('TOKEN:',TOKEN)
    print('\nMantén esta sesión de Colab abierta mientras uses el motor. No compartas estos datos.')


In [ ]:
# 5) Prueba rápida del motor
if url:
 import urllib.request
 payload=json.dumps({'model':'qwen2.5:3b-instruct-q4_K_M','messages':[{'role':'user','content':'Responde solo OK si el motor editorial está activo.'}],'stream':False}).encode()
 req=urllib.request.Request('http://127.0.0.1:8000/api/chat',data=payload,headers={'Content-Type':'application/json','Authorization':'Bearer '+TOKEN},method='POST')
 with urllib.request.urlopen(req,timeout=120) as response: print(json.loads(response.read())['message']['content'])
